# 02 - Preprocessing and baseline model

This notebook maps to CRISP-DM stages: **data preparation -> modeling -> evaluation**.

Main pitfalls to avoid:
- data leakage (preprocessing using validation/test data)
- biased splits
- reporting only training performance

**Goal:** produce a clean baseline and a validation RMSE you can explain.

In [ ]:
import os, sys
# Colab setup: make sure workshop source files are available.
if not os.path.exists("/content/citm1b"):
    !git clone https://github.com/MwiseG/citm1b.git
%cd /content/citm1b/worksop1_class

# Ensure `src` package can be imported reliably.
if "/content/citm1b/worksop1_class" not in sys.path:
    sys.path.insert(0, "/content/citm1b/worksop1_class")

!pwd
!ls

In [ ]:
from src.data_loading import load_housing_data
from src.preprocessing import stratified_split

# Load full dataset.
data, target = load_housing_data()

# Split into train/validation/test with stratification.
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(data, target)

# Check shapes to confirm split worked as expected.
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

### Why stratification on income?

`MedInc` is strongly related to house value. Stratifying by income category helps keep train/validation/test distributions similar and reduces sampling bias.

Without this, one split may accidentally contain mostly low- or high-income areas, making evaluation misleading.

In [ ]:
from src.preprocessing import build_preprocessing_pipeline

numeric_features = list(X_train.columns)
preprocessor = build_preprocessing_pipeline(numeric_features)

# Fit on training data only to avoid leakage.
X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

# Shapes should keep row counts and transformed feature columns.
print("Prepared train:", X_train_prepared.shape)
print("Prepared val:", X_val_prepared.shape)
print("Prepared test:", X_test_prepared.shape)

In [ ]:
import numpy as np

# After scaling, means should be near 0 and stds near 1 (on training data).
print("Approx feature means (first 5):", np.round(X_train_prepared.mean(axis=0)[:5], 3))
print("Approx feature stds (first 5):", np.round(X_train_prepared.std(axis=0)[:5], 3))

In [ ]:
from src.models import compute_rmse, train_baseline_linear_regression

# Train a simple baseline model.
baseline_model = train_baseline_linear_regression(X_train_prepared, y_train)

# Evaluate on train and validation to compare fit quality.
train_rmse = compute_rmse(baseline_model, X_train_prepared, y_train)
val_rmse = compute_rmse(baseline_model, X_val_prepared, y_val)

print("Train RMSE:", round(train_rmse, 4))
print("Validation RMSE:", round(val_rmse, 4))

### Overfitting and underfitting (plain language)

- If training RMSE is much better than validation RMSE, your model may be overfitting.
- If both are poor, your model may be underfitting.
- Baselines are useful references: simple first, then improve gradually.

Focus on the gap between train and validation, not only one number.

## Presentation prep

Each group should prepare:

1. One plot from `01_exploration.ipynb`.
2. Your validation RMSE from this notebook.
3. 1-2 sentences about one pitfall you tried to avoid (or a mistake you made) and how you fixed it.

LLMs are allowed: if you use one, be ready to explain the code in your own words.

Tip: keep your explanation concrete (what happened, why, and what you changed).

## Extra task (ML engineer practice): Save and reload artifacts

Training is only part of the job. Engineers must save preprocessing + model artifacts for reuse.

### TODO 6

1. Save `preprocessor` and `baseline_model` to disk using `joblib`.
2. Reload both objects.
3. Run inference on the first 5 rows from `X_test` using the reloaded objects.
4. Print the predictions.

Why this matters: this is a basic step toward reproducible deployment workflows.

In [ ]:
# YOUR CODE HERE (TODO 6)

# Suggested structure:
# import joblib
# joblib.dump(preprocessor, "preprocessor.joblib")
# joblib.dump(baseline_model, "baseline_model.joblib")
# loaded_preprocessor = joblib.load("preprocessor.joblib")
# loaded_model = joblib.load("baseline_model.joblib")
# sample_prepared = loaded_preprocessor.transform(X_test.head(5))
# sample_preds = loaded_model.predict(sample_prepared)
# sample_preds